# Customer Risk Scoring Analysis

## Project Overview
Analyze customer financial and behavioral data to understand how risk indicators vary across segments and time. This is a descriptive analytics project — no predictive modeling.

## Learning Objectives
- Profile customers by financial and behavioral risk indicators
- Construct simple heuristic risk scores and analyze their distributions
- Identify high-risk customer segments and their characteristics
- Track risk trends over time to detect portfolio deterioration

## Problem Statement
A financial services firm wants to understand its customer risk profile: Which segments carry the most risk? Are risk indicators worsening? What characteristics distinguish high-risk from low-risk customers?

## Why This Project Matters
Proactive risk monitoring prevents losses, enables early intervention, and supports regulatory requirements for portfolio risk assessment.

## Dataset Overview
Synthetic customer portfolio: ~4K customers with financial metrics, account behavior, and computed risk indicators.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 4000
segments = np.random.choice(['Retail', 'SME', 'Corporate', 'High Net Worth'], n, p=[0.40, 0.30, 0.15, 0.15])
regions = np.random.choice(['East', 'West', 'North', 'South'], n, p=[0.28, 0.25, 0.22, 0.25])

# Financial indicators
credit_score = np.clip(np.random.normal(680, 60, n).astype(int), 400, 850)
annual_income = np.clip(np.random.lognormal(11.0, 0.6, n), 20000, 2000000).round(0)
total_balance = np.clip(np.random.lognormal(9.5, 1.2, n), 500, 500000).round(2)
utilization = np.clip(np.random.beta(2, 5, n) * 100, 0, 100).round(1)

# Behavioral indicators
months_since_last_late = np.clip(np.random.exponential(12, n).astype(int), 0, 60)
num_products = np.random.poisson(3, n) + 1
tenure_months = np.clip(np.random.exponential(36, n).astype(int), 1, 240)
complaints_12m = np.random.poisson(0.5, n)

# Onboarding date
onboard_dates = pd.date_range('2018-01-01', '2024-06-01', freq='D')
onboard = np.array([pd.Timestamp(d) for d in np.random.choice(onboard_dates, n)])

# Heuristic risk score (0-100, higher = riskier)
risk_score = np.clip(
    50
    - (credit_score - 600) * 0.08
    + utilization * 0.3
    - months_since_last_late * 0.5
    + complaints_12m * 5
    + np.random.normal(0, 5, n),
    0, 100
).round(1)

df = pd.DataFrame({
    'CustomerID': [f'C{i:05d}' for i in range(n)],
    'Segment': segments, 'Region': regions,
    'CreditScore': credit_score, 'AnnualIncome': annual_income,
    'TotalBalance': total_balance, 'Utilization': utilization,
    'MonthsSinceLastLate': months_since_last_late,
    'NumProducts': num_products, 'TenureMonths': tenure_months,
    'Complaints12M': complaints_12m,
    'OnboardDate': onboard,
    'RiskScore': risk_score,
})
df['RiskTier'] = pd.cut(df['RiskScore'], bins=[0, 25, 50, 75, 100],
                         labels=['Low', 'Medium', 'High', 'Critical'])
df['OnboardYear'] = df['OnboardDate'].dt.year

print(f'Dataset shape: {df.shape}')
print(f'Avg risk score: {df["RiskScore"].mean():.1f}')
print(f'Risk tier distribution:\n{df["RiskTier"].value_counts().sort_index()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Credit score range: {df["CreditScore"].min()} - {df["CreditScore"].max()}')
print(f'Risk score range: {df["RiskScore"].min()} - {df["RiskScore"].max()}')
print(f'\nSegment distribution:\n{df["Segment"].value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df['RiskScore'].hist(bins=40, ax=axes[0,0], edgecolor='black', alpha=0.7, color='coral')
axes[0,0].set_title('Risk Score Distribution')
axes[0,0].axvline(df['RiskScore'].mean(), color='red', linestyle='--', label=f'Mean: {df["RiskScore"].mean():.1f}')
axes[0,0].legend()

df.groupby('Segment')['RiskScore'].mean().sort_values().plot.barh(ax=axes[0,1], edgecolor='black', color='steelblue')
axes[0,1].set_title('Avg Risk Score by Segment')

df.groupby('RiskTier')['TotalBalance'].sum().plot.bar(ax=axes[1,0], edgecolor='black', color='green')
axes[1,0].set_title('Total Exposure by Risk Tier')
axes[1,0].set_ylabel('Total Balance ($)')
axes[1,0].tick_params(axis='x', rotation=0)

sns.scatterplot(data=df.sample(500, random_state=42), x='CreditScore', y='RiskScore',
                hue='RiskTier', alpha=0.5, ax=axes[1,1])
axes[1,1].set_title('Credit Score vs Risk Score')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Risk Profile by Segment

In [ ]:
seg_profile = df.groupby('Segment').agg(
    count=('CustomerID', 'count'),
    avg_risk=('RiskScore', 'mean'),
    high_risk_pct=('RiskTier', lambda x: (x.isin(['High', 'Critical'])).mean()),
    avg_credit=('CreditScore', 'mean'),
    avg_utilization=('Utilization', 'mean'),
    avg_balance=('TotalBalance', 'mean'),
    total_exposure=('TotalBalance', 'sum'),
).round(3).sort_values('avg_risk', ascending=False)
print('Risk Profile by Segment:')
print(seg_profile)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
seg_profile['avg_risk'].sort_values().plot.barh(ax=axes[0], edgecolor='black', color='coral')
axes[0].set_title('Avg Risk Score by Segment')

seg_profile['high_risk_pct'].sort_values().plot.barh(ax=axes[1], edgecolor='black', color='red', alpha=0.6)
axes[1].set_title('High/Critical Risk % by Segment')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'segment_risk.png'), dpi=100, bbox_inches='tight')
plt.show()

## Risk by Region

In [ ]:
reg_risk = df.groupby('Region').agg(
    avg_risk=('RiskScore', 'mean'),
    critical_pct=('RiskTier', lambda x: (x == 'Critical').mean()),
    total_exposure=('TotalBalance', 'sum'),
).round(3).sort_values('avg_risk', ascending=False)
print('Regional Risk:')
print(reg_risk)

fig, ax = plt.subplots(figsize=(10, 5))
reg_risk['avg_risk'].plot.bar(ax=ax, edgecolor='black', color='mediumpurple')
ax.set_title('Avg Risk Score by Region')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'regional_risk.png'), dpi=100, bbox_inches='tight')
plt.show()

## Risk Indicator Correlation

In [ ]:
risk_cols = ['RiskScore', 'CreditScore', 'Utilization', 'MonthsSinceLastLate', 'Complaints12M', 'TenureMonths']
corr = df[risk_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Risk Indicator Correlations')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'correlation.png'), dpi=100, bbox_inches='tight')
plt.show()

## Onboarding Cohort Risk Comparison

In [ ]:
cohort = df.groupby('OnboardYear').agg(
    customers=('CustomerID', 'count'),
    avg_risk=('RiskScore', 'mean'),
    high_risk_pct=('RiskTier', lambda x: (x.isin(['High', 'Critical'])).mean()),
).round(3)
print('Risk by Onboarding Cohort:')
print(cohort)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(cohort.index, cohort['avg_risk'], edgecolor='black', color='teal', alpha=0.7)
ax.set_title('Avg Risk Score by Onboarding Year')
ax.set_xlabel('Onboarding Year')
ax.set_ylabel('Avg Risk Score')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'cohort_risk.png'), dpi=100, bbox_inches='tight')
plt.show()

## High-Risk Customer Characteristics

In [ ]:
high_risk = df[df['RiskTier'].isin(['High', 'Critical'])]
low_risk = df[df['RiskTier'] == 'Low']
compare_cols = ['CreditScore', 'AnnualIncome', 'TotalBalance', 'Utilization',
                'MonthsSinceLastLate', 'NumProducts', 'TenureMonths', 'Complaints12M']
comparison = pd.DataFrame({
    'Low Risk': low_risk[compare_cols].mean(),
    'High/Critical': high_risk[compare_cols].mean(),
}).round(2)
comparison['Ratio'] = (comparison['High/Critical'] / comparison['Low Risk']).round(2)
print('High-Risk vs Low-Risk Profile:')
print(comparison)

## Interpretation and Business Insight
- **Credit score** and **utilization** are the strongest risk indicators — customers with high utilization and low credit scores dominate the Critical tier
- **Retail** and **SME** segments carry more risk on average than Corporate/HNW
- **Recent complaints** strongly predict higher risk — a behavioral early warning signal
- **Tenure** shows a modest protective effect — longer relationships correlate with lower risk
- **Onboarding cohort** analysis reveals whether newer customers are riskier (vintage analysis)
- The heuristic risk score effectively separates customer profiles

## Limitations
- Synthetic data with a pre-defined risk formula — real risk scoring uses validated models
- No default/loss outcome data to validate the risk score
- Static snapshot — real risk changes continuously
- No macroeconomic context (recession, boom periods)
- Heuristic score is not a proper credit model

## How to Improve This Project
- Use real portfolio data with actual default outcomes
- Build validated credit scoring models (logistic regression, gradient boosting)
- Add stress testing and scenario analysis
- Include macroeconomic overlays for forward-looking risk assessment
- Implement monitoring dashboards with early warning triggers

## Production Considerations
- Automate risk score updates on a monthly cycle
- Build alerts for customers crossing risk tier thresholds
- Track portfolio-level risk concentration limits
- Integrate with decisioning systems for credit line management

## Common Mistakes
- Using risk indicators without understanding their causal direction
- Ignoring correlation between risk factors (multicollinearity in scoring models)
- Treating the risk score as absolute — it's only a relative ranking
- Not validating the score against actual outcomes (discrimination and calibration)

## Mini Challenge / Exercises
1. What percentage of total balance (exposure) is concentrated in the Critical risk tier?
2. Which segment has the highest concentration risk (highest % of total exposure in High/Critical)?
3. Create an alternative risk score using only CreditScore and Utilization. How does it compare?
4. If you moved all customers with Complaints12M > 2 up one risk tier, how much additional exposure falls into Critical?

## Final Summary / Key Takeaways
- Customer risk profiling combines financial strength (credit score, income) with behavioral signals (utilization, complaints)
- Segment-level analysis reveals systematic risk differences that inform credit policy
- Cohort and trend analysis detect whether portfolio risk is improving or deteriorating
- Simple heuristic scores are a powerful first step, but validated models are essential for production
- Risk management requires continuous monitoring, not one-time analysis